# Credit Score Multi Class Classification

In [ ]:
import pandas as pd

train_data = pd.read_csv('../dataset/train.csv')
test_data = pd.read_csv('../dataset/test.csv')

print("Train Data Shape:", train_data.shape)
print("Test Data Shape:", test_data.shape)

train_data.head()

C:\Users\adiln\AppData\Local\Temp\ipykernel_26776\3792723488.py:3: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv('../dataset/train.csv')


Train Data Shape: (100000, 28)
Test Data Shape: (50000, 27)


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


In [ ]:
train_data.dtypes

ID                           object
Customer_ID                  object
Month                        object
Name                         object
Age                          object
SSN                          object
Occupation                   object
Annual_Income                object
Monthly_Inhand_Salary       float64
Num_Bank_Accounts             int64
Num_Credit_Card               int64
Interest_Rate                 int64
Num_of_Loan                  object
Type_of_Loan                 object
Delay_from_due_date           int64
Num_of_Delayed_Payment       object
Changed_Credit_Limit         object
Num_Credit_Inquiries        float64
Credit_Mix                   object
Outstanding_Debt             object
Credit_Utilization_Ratio    float64
Credit_History_Age           object
Payment_of_Min_Amount        object
Total_EMI_per_month         float64
Amount_invested_monthly      object
Payment_Behaviour            object
Monthly_Balance              object
Credit_Score                

Splitting the data into training and validation sets.

In [3]:
from sklearn.model_selection import train_test_split

X = train_data.drop('Credit_Score', axis=1)
y = train_data['Credit_Score']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train Shape:", X_train.shape)
print("X_val Shape:", X_val.shape)
print("y_train Shape:", y_train.shape)
print("y_val Shape:", y_val.shape)

X_train Shape: (80000, 27)
X_val Shape: (20000, 27)
y_train Shape: (80000,)
y_val Shape: (20000,)


Data Preprocessing and Feature Engineering steps

In [4]:
# Checking for missing values and data types
print("Missing values:\n", X_train.isnull().sum())

# Checking for categorical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"\nNumerical columns ({len(numerical_cols)}): {numerical_cols}")

Missing values:
 ID                              0
Customer_ID                     0
Month                           0
Name                         8057
Age                             0
SSN                             0
Occupation                      0
Annual_Income                   0
Monthly_Inhand_Salary       11986
Num_Bank_Accounts               0
Num_Credit_Card                 0
Interest_Rate                   0
Num_of_Loan                     0
Type_of_Loan                 9145
Delay_from_due_date             0
Num_of_Delayed_Payment       5602
Changed_Credit_Limit            0
Num_Credit_Inquiries         1587
Credit_Mix                      0
Outstanding_Debt                0
Credit_Utilization_Ratio        0
Credit_History_Age           7234
Payment_of_Min_Amount           0
Total_EMI_per_month             0
Amount_invested_monthly      3585
Payment_Behaviour               0
Monthly_Balance               942
dtype: int64
Categorical columns (19): ['ID', 'Customer_ID', 'Mon

In [14]:
# Preprocessing Pipeline
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Create copies to avoid modifying original data
X_train_processed = X_train.copy()
X_val_processed = X_val.copy()

# Step 1: Drop ID columns that don't contribute to prediction
id_cols = ['ID', 'Customer_ID', 'Name', 'SSN']
X_train_processed = X_train_processed.drop(columns=[col for col in id_cols if col in X_train_processed.columns])
X_val_processed = X_val_processed.drop(columns=[col for col in id_cols if col in X_val_processed.columns])

# Step 2: Handle columns with mixed types (like Age which is object but should be numeric)
def clean_numeric_column(series):
    """Convert object columns with numeric values to float"""
    if series.dtype == 'object':
        # Remove any non-numeric characters except decimal point and minus sign
        series = series.astype(str).str.replace('[^0-9.-]', '', regex=True)
        # Replace empty strings with NaN
        series = series.replace('', np.nan)
        # Convert to float
        try:
            series = pd.to_numeric(series, errors='coerce')
        except:
            pass
    return series

# Columns that should be numeric but are stored as object
numeric_object_cols = ['Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment', 
                       'Changed_Credit_Limit', 'Outstanding_Debt', 'Amount_invested_monthly', 
                       'Monthly_Balance']

for col in numeric_object_cols:
    if col in X_train_processed.columns:
        X_train_processed[col] = clean_numeric_column(X_train_processed[col])
        X_val_processed[col] = clean_numeric_column(X_val_processed[col])

# Step 3: Handle Credit_History_Age (e.g., "22 Years and 1 Months")
def parse_credit_history_age(series):
    """Convert credit history age to total months"""
    if series.dtype == 'object':
        # Extract years and months
        years = series.astype(str).str.extract(r'(\d+)\s*Years?', expand=False).astype(float)
        months = series.astype(str).str.extract(r'(\d+)\s*Months?', expand=False).astype(float)
        # Convert to total months
        total_months = years.fillna(0) * 12 + months.fillna(0)
        return total_months
    return series

if 'Credit_History_Age' in X_train_processed.columns:
    X_train_processed['Credit_History_Age'] = parse_credit_history_age(X_train_processed['Credit_History_Age'])
    X_val_processed['Credit_History_Age'] = parse_credit_history_age(X_val_processed['Credit_History_Age'])

# Step 4: Encode categorical variables
categorical_cols = X_train_processed.select_dtypes(include=['object']).columns.tolist()

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    
    # Fit on training data
    X_train_processed[col] = X_train_processed[col].fillna('Unknown')
    le.fit(X_train_processed[col])
    
    # Transform both train and validation
    X_train_processed[col] = le.transform(X_train_processed[col])
    
    # Handle unseen categories in validation set
    X_val_processed[col] = X_val_processed[col].fillna('Unknown')
    X_val_processed[col] = X_val_processed[col].apply(
        lambda x: x if x in le.classes_ else 'Unknown'
    )
    X_val_processed[col] = le.transform(X_val_processed[col])
    
    label_encoders[col] = le

# Step 5: Handle missing values in numerical columns
imputer = SimpleImputer(strategy='median')
X_train_processed = pd.DataFrame(
    imputer.fit_transform(X_train_processed),
    columns=X_train_processed.columns,
    index=X_train_processed.index
)
X_val_processed = pd.DataFrame(
    imputer.transform(X_val_processed),
    columns=X_val_processed.columns,
    index=X_val_processed.index
)

# Step 6: Scale features
scaler = StandardScaler()
X_train_processed = pd.DataFrame(
    scaler.fit_transform(X_train_processed),
    columns=X_train_processed.columns,
    index=X_train_processed.index
)
X_val_processed = pd.DataFrame(
    scaler.transform(X_val_processed),
    columns=X_val_processed.columns,
    index=X_val_processed.index
)

print("Preprocessing complete!")
print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_val_processed shape: {X_val_processed.shape}")
print(f"\nNo more object dtypes: {X_train_processed.select_dtypes(include=['object']).columns.tolist()}")
print(f"All columns are numeric: {X_train_processed.dtypes.unique()}")

Preprocessing complete!
X_train_processed shape: (80000, 23)
X_val_processed shape: (20000, 23)

No more object dtypes: []
All columns are numeric: [dtype('float64')]


# Logistic Regression

In [15]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef


model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_processed, y_train)
y_pred = model.predict(X_val_processed)

# Calculate metrics
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='weighted')
recall = recall_score(y_val, y_pred, average='weighted')
f1 = f1_score(y_val, y_pred, average='weighted')

# For multi-class classification, use label binarization for ROC AUC
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score as roc_auc_multiclass

y_val_binarized = label_binarize(y_val, classes=np.unique(y_train))
y_pred_proba = model.predict_proba(X_val_processed)
roc_auc = roc_auc_multiclass(y_val_binarized, y_pred_proba, average='weighted', multi_class='ovr')

# MCC for multiclass
mcc = matthews_corrcoef(y_val, y_pred)

print(f"Logistic Regression Results:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  ROC AUC:   {roc_auc:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  MCC:       {mcc:.4f}")

Logistic Regression Results:
  Accuracy:  0.6039
  ROC AUC:   0.7356
  Precision: 0.6016
  Recall:    0.6039
  F1 Score:  0.5867
  MCC:       0.3010
